# Find and define unique peptides for all isoforms
In this notebook we will use the results from the tryptic in silico digestion and try to define a unique peptide for every isofofrm in our proteome.

# Setup
## Import and install Packages

In [34]:
import sys
import os
import session_info

In [35]:
import pandas as pd

In [36]:
# Display session information
session_info.show()

## Set working directory

In [37]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/02_In-Silico_Digestion"):
    print("WARNING: The working directory is not set to the '02_In-Silico_Digestion'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_In-Silico_Digestion' folder (\"{cwd}\").")

Current working directory: /Users/lauranieba/Documents/Uni/Master_ETH/FS26/Lab_Rotation_Davos/Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_In-Silico_Digestion


In [38]:
# Data directory for fasta data
mapping_dir = "../01_UniProt/data/mapping/"
digestion_output_dir = "data/digestion_output"

## Read in Dataframes

In [39]:
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

digestion_full_proteome_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_full_proteome_filtered.csv'))

# Isoform-specific peptides

In [40]:
# 1. Separate your digestion into two buckets
# Canonical IDs (already stripped of -1) vs Isoform IDs (usually -2, -3, etc.)
canonical_ids = iso_canonical_mapping["Canonical_Isoform_ID"].dropna().tolist()
canonical_base_ids = [cid[:-2] if cid.endswith('-1') else cid for cid in canonical_ids]

canonical_peptides = set(digestion_full_proteome_filtered[
    digestion_full_proteome_filtered['ID'].isin(canonical_base_ids)
]['seq'])

isoform_df = digestion_full_proteome_filtered[
    ~digestion_full_proteome_filtered['ID'].isin(canonical_base_ids)
].copy()

# 2. Identify peptides in isoforms that are NEVER found in any canonical protein
all_isoform_peptides = set(isoform_df['seq'])
isoform_specific_peptides = all_isoform_peptides - canonical_peptides

# 3. Filter the isoform dataframe to keep only these specific identifiers
isoform_unique_db = isoform_df[isoform_df['seq'].isin(isoform_specific_peptides)].copy()

# 4. Optional: Ensure the peptide is unique to ONLY ONE isoform 
# (In case two isoforms share the same mutation)
final_counts = isoform_unique_db.groupby('seq')['ID'].nunique()
truly_unique_seqs = final_counts[final_counts == 1].index

isoform_unique_db = isoform_unique_db[isoform_unique_db['seq'].isin(truly_unique_seqs)]

In [41]:
isoform_unique_db

,ID,seqID,seq,len
0,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,MGLEALVPLAMIVAIFLLLVDLMHR,25
3,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,GEDTADRPPAPIYQVLGFGPR,21
4,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,SQGVILSR,8
6,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,SLEQWVTEEAACLCAAFADQAGRPFRPNGLLDK,33
11,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,EVLNAVPVLPHIPALAGK,18
...,...,...,...,...
1223215,E9PI22_P0DMB1,sp|E9PI22_P0DMB1|P23D1_P23D2_HUMAN,ENVPFSPAEEGK,12
1223216,E9PI22_P0DMB1,sp|E9PI22_P0DMB1|P23D1_P23D2_HUMAN,AAPLYQQPLMIPQANHMAGISPSFLVTPLCIPR,33
1223217,E9PI22_P0DMB1,sp|E9PI22_P0DMB1|P23D1_P23D2_HUMAN,AAFPQCYPLPPTPSPVGRPRPADSSFSLHGMELLCTSSLRPMPPSP...,55
1223218,E9PI22_P0DMB1,sp|E9PI22_P0DMB1|P23D1_P23D2_HUMAN,VHHRPPSR,8


In [42]:
# Create a mapping for your database
lookup_table = isoform_unique_db[['ID', 'seq', 'len']].drop_duplicates()

# Sort by length if you prefer longer peptides for better MS/MS confidence
lookup_table = lookup_table.sort_values(['ID', 'len'], ascending=[True, False])

In [43]:
lookup_table

,ID,seq,len
705919,A0A024R1R8,MSSHEGGK,8
705920,A0A024R1R8,EMDEEEK,7
543491,A0A075B6H7,ASQSVSSSYLTWYQQKPGQAPR,22
543489,A0A075B6H7,MEAPAQLLFLLLLWLPDTTR,20
543490,A0A075B6H7,EIVMTQSPPTLSLSPGER,18
...,...,...,...
543488,W6CW81,LGLSFHGISGNAC,13
543485,W6CW81,WHGVCSEEDR,10
543486,W6CW81,LNYMLVAK,8
1222521,X6R8D5-2,EHESPSQPHMCGWEDSQKPSVPSHGPK,27


# Slightly different implementaton to flag isoforms for which we can't find a unique peptide like this
Insertions/Alternative Exons: These almost always produce unique peptides.
Deletions: These create a new "junction" peptide (where two parts of the protein are joined that aren't joined in the canonical).
Truncations: These often produce no unique peptides because the isoform is just a shorter version of the original.

Implementation: The "Best Candidate" Selection
Instead of just filtering, we will rank peptides for every isoform. This ensures that if a unique peptide exists, you grab it; if not, you flag that isoform for a different enzyme.

In [44]:
# 1. Create a "Global Peptide Map" (How many/which IDs have this peptide?)
# This tells us exactly which IDs share a sequence
peptide_to_ids = digestion_full_proteome_filtered.groupby('seq')['ID'].apply(set).to_dict()

# 2. Map every Isoform to its list of peptides
isoform_to_peptides = digestion_only_isoforms.groupby('ID')['seq'].apply(list).to_dict()

final_results = []

for iso_id, peptides in isoform_to_peptides.items():
    unique_to_this_isoform = []
    
    for p in peptides:
        # A peptide is a 'Perfect Match' if it belongs ONLY to this ID
        if peptide_to_ids[p] == {iso_id}:
            unique_to_this_isoform.append(p)
            
    if unique_to_this_isoform:
        # If multiple exist, pick the longest one (better for Mass Spec)
        best_peptide = max(unique_to_this_isoform, key=len)
        status = "Unique Found"
    else:
        best_peptide = None
        status = "No Unique Peptide (Shared/Truncated)"
        
    final_results.append({
        "Isoform_ID": iso_id,
        "Unique_Peptide": best_peptide,
        "Status": status
    })

results_df = pd.DataFrame(final_results)

In [45]:
results_df

,Isoform_ID,Unique_Peptide,Status
0,A0A024R1R8,MSSHEGGK,Unique Found
1,A0A075B6H7,ASQSVSSSYLTWYQQKPGQAPR,Unique Found
2,A0A075B6H8,VSIICWASEGISSNLAWYLQKPGK,Unique Found
3,A0A075B6H9,MAWTPLLFLTLLLHCTGSLSQLVLTQSPSASASLGASVK,Unique Found
4,A0A075B6I0,MSVPTMAWMMLLLGLLAYGSGVDSQTVVTQEPSFSVSPGGTVTLTC...,Unique Found
...,...,...,...
32293,W5XKT8-2,METQEYGESER,Unique Found
32294,W5XKT8-3,EDGESGGWR,Unique Found
32295,W6CW81,EILLLTSLDNITDEELDR,Unique Found
32296,X6R8D5-2,EHESPSQPHMCGWEDSQKPSVPSHGPK,Unique Found


In [46]:
print(results_df['Status'].value_counts())

Status
Unique Found                            24571
No Unique Peptide (Shared/Truncated)     7727
Name: count, dtype: int64
